# Etherpad Pad 内容重建查看器

本工具用于：
1. 查询数据库获取所有 room-id 和版本范围
2. SSH 连接到服务器执行重建脚本
3. 获取并展示重建的内容
4. 支持编辑和对比

---

## 📋 功能清单

- ✅ 查询 MySQL 数据库获取所有 Pad 列表
- ✅ 显示每个 Pad 的版本范围
- ✅ SSH 连接到远程服务器执行重建脚本
- ✅ 查看pad_changes详细内容


## 使用说明

### 完整使用流程

1. **安装依赖** 
2. **导入配置** 
3. **定义函数** 
4. **定义http stream函数** 
5. **初步查询数据库的ROOM基本请开给你** 
6. **分两种方法跑重建的数据**：数据来源于store表，跑线上http stream api重建数据<br>
   6.1.限制pad_id和起始版本号跑重建数据<br>
   6.2.限制起始时间范围跑重建数据
7. **查询的是pad_changes表**：pad_changes记录的每个文档发生增加删除行为的数据

## 1. 安装依赖包

首次使用需要安装必要的 Python 包


In [1]:
# 安装必要的 Python 包
#!pip install pymysql paramiko pandas ipywidgets -q


## 2. 导入库和配置


In [2]:
import pymysql
import json
import pandas as pd
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
import re
from datetime import datetime
from collections import defaultdict
import urllib.request
import urllib.parse
import json
import requests

# 数据库配置
DB_CONFIG = {
    'host': '112.74.92.135',
    'user': 'root',
    'password': '1q2w3e4R',
    'database': 'alic',
    'charset': 'utf8mb4',
    'port': 3306
}

# HTTP API 配置 - 修复版本（直接使用HTTP连接到服务器IP）
# 问题说明：域名 etherpad-test.alicedu.net 解析到 VPN 网关 198.18.0.77
# VPN 网关的 SSL 配置有问题，导致 SSL 握手失败
# 解决方案：直接使用 HTTP 协议连接到服务器真实 IP
HTTP_API_CONFIG = {
    'host': '112.74.92.135',  # 直接使用服务器真实IP
    'port': 9002,            # HTTP端口
    'protocol': 'http'       # 使用HTTP协议（绕过有问题的SSL网关）
}

print("✅ 配置加载完成")
print("💡 使用 HTTP 直接连接到服务器 112.74.92.135")
print("💡 绕过 VPN 网关 112.74.92.135 的 SSL 问题")


✅ 配置加载完成
💡 使用 HTTP 直接连接到服务器 112.74.92.135
💡 绕过 VPN 网关 112.74.92.135 的 SSL 问题


## 3. 定义数据库查询函数


In [3]:
def get_all_room_ids_with_versions():
    """从 store 表查询所有 room-id 及其版本范围"""
    try:
        connection = pymysql.connect(**DB_CONFIG)
        cursor = connection.cursor(pymysql.cursors.DictCursor)
        
        query = """
        SELECT 
            SUBSTRING_INDEX(SUBSTRING_INDEX(`key`, ':', 2), ':', -1) as pad_id,
            MIN(CAST(SUBSTRING_INDEX(`key`, ':', -1) AS UNSIGNED)) as min_revision,
            MAX(CAST(SUBSTRING_INDEX(`key`, ':', -1) AS UNSIGNED)) as max_revision,
            COUNT(*) as revision_count
        FROM store
        WHERE `key` LIKE 'pad:%:revs:%'
        GROUP BY pad_id
        ORDER BY pad_id
        """
        
        cursor.execute(query)
        results = cursor.fetchall()
        
        cursor.close()
        connection.close()
        
        return results
        
    except Exception as e:
        print(f"❌ 数据库查询失败: {e}")
        return []

print("✅ 数据库查询函数定义完成")


def query_pads_by_timerange(start_time, end_time):
    """
    根据时间范围查询 Store 数据库中的 Pad 版本信息
    
    参数:
        start_time: 开始时间字符串，格式 'YYYY-MM-DD HH:MM:SS'
        end_time: 结束时间字符串，格式 'YYYY-MM-DD HH:MM:SS'
    
    返回:
        dict: {pad_id: [revision1, revision2, ...]}
    """
    # 转换为时间戳（毫秒）
    start_timestamp = int(datetime.strptime(start_time, '%Y-%m-%d %H:%M:%S').timestamp() * 1000)
    end_timestamp = int(datetime.strptime(end_time, '%Y-%m-%d %H:%M:%S').timestamp() * 1000)
    
    print(f"🔍 查询时间范围: {start_time} ~ {end_time}")
    print(f"📊 时间戳范围: {start_timestamp} ~ {end_timestamp}\n")
    
    # 查询 Store 数据库
    print("🔍 正在查询 Store 数据库...")
    pad_results = []
    
    try:
        connection = pymysql.connect(**DB_CONFIG)
        cursor = connection.cursor(pymysql.cursors.DictCursor)

        query = """
        SELECT `key`, `value`
        FROM `store`
        WHERE `key` LIKE 'pad:%%:revs:%%'
          AND JSON_EXTRACT(`value`, '$.meta.timestamp') >= %s
          AND JSON_EXTRACT(`value`, '$.meta.timestamp') <= %s
        ORDER BY `key`
        """
        
        cursor.execute(query, (start_timestamp, end_timestamp))
        pad_results = cursor.fetchall()
        
        print(f"✅ 找到 {len(pad_results)} 条记录\n")
        
        cursor.close()
        connection.close()
        
    except Exception as e:
        print(f"❌ 数据库查询失败: {e}\n")
        return {}
    
    # 解析 pad_id 和版本号
    print("📊 正在解析 pad_id 和版本号...")
    
    # 正则表达式：pad:room-id:revs:revision
    pattern = re.compile(r'^pad:([^:]+):revs:(\d+)$')
    pad_versions = defaultdict(list)
    
    for row in pad_results:
        key = row['key']
        match = pattern.match(key)
        if match:
            pad_id = match.group(1)
            revision = int(match.group(2))
            pad_versions[pad_id].append(revision)
    
    print(f"✅ 解析完成，共 {len(pad_versions)} 个不同的 Pad")
    
    # 显示每个 Pad 的版本范围
    print("📋 Pad 版本范围统计:")
    for pad_id, revisions in sorted(pad_versions.items()):
        revisions.sort()
        print(f"  - {pad_id}: 版本 {min(revisions)} ~ {max(revisions)} (共 {len(revisions)} 个版本)")
    
    return dict(pad_versions)

✅ 数据库查询函数定义完成


## 4. 定义 SSH 远程执行函数


In [4]:
import base64
import urllib.request
import urllib.parse

def decode_base64_field(value):
    """解码 Base64 字段"""
    if not value:
        return ''
    try:
        return base64.b64decode(value).decode('utf-8')
    except Exception:
        return value  # 如果解码失败，返回原值

def execute_rebuild_script(pad_id, start_rev=None, end_rev=None, use_base64=False, batch_size=None):
    """
    通过 HTTP Stream 获取重建数据（支持自动分批处理）
    
    参数:
        pad_id: Pad ID
        start_rev: 起始版本号
        end_rev: 结束版本号
        use_base64: 是否使用 Base64 编码模式（推荐用于大数据量）
        batch_size: 批次大小（如果指定，则自动分批处理）
    """
    
    # 如果指定了 batch_size，则进行分批处理
    if batch_size and start_rev is not None and end_rev is not None:
        total_versions = end_rev - start_rev + 1
        
        # 如果版本数小于等于批次大小，直接处理
        if total_versions <= batch_size:
            return _execute_single_rebuild(pad_id, start_rev, end_rev, use_base64)
        
        # 需要分批处理
        num_batches = (total_versions + batch_size - 1) // batch_size
        
        print(f"📊 总共 {total_versions} 个版本，将分 {num_batches} 个批次处理（每批 {batch_size} 个）\n")
        
        all_versions = []
        total_success = 0
        total_failed = 0
        
        for batch_idx in range(num_batches):
            batch_start = start_rev + batch_idx * batch_size
            batch_end = min(batch_start + batch_size - 1, end_rev)
            
            print(f"🔄 批次 {batch_idx + 1}/{num_batches}: 版本 {batch_start} - {batch_end}")
            
            # 执行单个批次
            batch_result = _execute_single_rebuild(pad_id, batch_start, batch_end, use_base64)
            
            if batch_result.get('success'):
                batch_versions = batch_result.get('versions', [])
                all_versions.extend(batch_versions)
                
                batch_stats = batch_result.get('statistics', {})
                batch_success = batch_stats.get('success', len(batch_versions))
                batch_failed = batch_stats.get('failed', 0)
                
                total_success += batch_success
                total_failed += batch_failed
                
                print(f"   ✅ 成功: {batch_success}, 失败: {batch_failed}\n")
            else:
                error_msg = batch_result.get('error', 'Unknown error')
                print(f"   ❌ 批次失败: {error_msg}")
                total_failed += (batch_end - batch_start + 1)
        
        # 返回合并后的结果
        return {
            'success': True,
            'pad_id': pad_id,
            'head_revision': end_rev,
            'requested_range': {
                'start': start_rev,
                'end': end_rev
            },
            'statistics': {
                'total': total_versions,
                'success': total_success,
                'failed': total_failed
            },
            'versions': all_versions,
            'batch_info': {
                'batch_size': batch_size,
                'num_batches': num_batches
            }
        }
    else:
        # 不分批，直接处理
        return _execute_single_rebuild(pad_id, start_rev, end_rev, use_base64)


def _execute_single_rebuild(pad_id, start_rev=None, end_rev=None, use_base64=False):
    """
    执行单次重建请求（内部函数）- 使用 requests 库
    
    参数:
        pad_id: Pad ID
        start_rev: 起始版本号
        end_rev: 结束版本号
        use_base64: 是否使用 Base64 编码模式
    """
    try:
        # 构建 HTTP 请求参数
        params = {
            'padId': pad_id,
            'useBase64': 'true' if use_base64 else 'false'
        }
        
        if start_rev is not None:
            params['startRev'] = str(start_rev)
        if end_rev is not None:
            params['endRev'] = str(end_rev)

        protocol = HTTP_API_CONFIG.get('protocol', 'http')
        port = HTTP_API_CONFIG['port']
        host = HTTP_API_CONFIG['host']
        
        # 构建URL
        if (protocol == 'https' and port == 443) or (protocol == 'http' and port == 80):
            url = f"{protocol}://{host}/api/rebuild"
        else:
            url = f"{protocol}://{host}:{port}/api/rebuild"
        
        # 使用 requests 发送请求（流式读取）
        response = requests.get(
            url, 
            params=params, 
            stream=True,  # 流式读取
            timeout=600   # 10 分钟超时
        )
        
        # 检查响应状态
        response.raise_for_status()
        
        # 解析 NDJSON 流式响应
        all_versions = []
        header_info = None
        statistics = None
        
        # 逐行读取 NDJSON 格式的响应
        for line in response.iter_lines():
            if not line:
                continue
            
            try:
                data = json.loads(line.decode('utf-8'))
                
                # 第一行是头部信息
                if 'stream' in data and data.get('stream'):
                    header_info = data
                    continue
                
                # 最后一行是统计信息
                if '_statistics' in data:
                    statistics = data.get('_statistics', {})
                    continue
                
                # 版本数据
                if 'revision' in data:
                    all_versions.append(data)
                    
            except json.JSONDecodeError:
                continue
        
        # 构建结果对象
        result = {
            'success': True,
            'pad_id': pad_id,
            'head_revision': header_info.get('head_revision') if header_info else None,
            'requested_range': header_info.get('requested_range', {}) if header_info else {},
            'statistics': statistics or {},
            'versions': all_versions
        }
        
        # 处理 Base64 解码
        if header_info and header_info.get('encoding') == 'base64':
            result['encoding'] = 'base64'
            for version in all_versions:
                if 'content_base64' in version:
                    version['content'] = decode_base64_field(version['content_base64'])
                    del version['content_base64']
                if 'changeset_base64' in version:
                    version['changeset'] = decode_base64_field(version['changeset_base64'])
                    del version['changeset_base64']
                if 'attribs_base64' in version:
                    version['attribs'] = decode_base64_field(version['attribs_base64'])
                    del version['attribs_base64']
        
        return result
        
    except requests.exceptions.HTTPError as e:
        return {'success': False, 'error': f'HTTP 错误 {e.response.status_code}: {e.response.text}'}
    
    except requests.exceptions.ConnectionError as e:
        return {'success': False, 'error': f'连接错误: {str(e)}'}
    
    except requests.exceptions.Timeout as e:
        return {'success': False, 'error': f'请求超时: {str(e)}'}
        
    except Exception as e:
        import traceback
        traceback.print_exc()
        return {'success': False, 'error': str(e)}


print("✅ HTTP Stream 执行函数定义完成（支持自动分批处理）")

✅ HTTP Stream 执行函数定义完成（支持自动分批处理）


## 5. 查询所有 Room ID 和版本范围

初步查一查文档的基本情况

In [5]:
# 查询所有 room-id 及版本范围
print("🔍 正在查询数据库...")
room_list = get_all_room_ids_with_versions()

if room_list:
    df = pd.DataFrame(room_list)
    df.index = df.index + 1
    
    print(f"\n✅ 找到 {len(room_list)} 个 Pad\n")
    display(df)
    
    # 统计信息
    total_revisions = df['revision_count'].sum()
    avg_revisions = df['revision_count'].mean()
    
    print(f"\n📊 统计信息:")
    print(f"   总 Pad 数: {len(room_list)}")
    print(f"   总版本数: {total_revisions}")
    print(f"   平均版本数: {avg_revisions:.2f}")
    print(f"   最多版本: {df['revision_count'].max()} (Pad: {df.loc[df['revision_count'].idxmax(), 'pad_id']})")
    print(f"   最少版本: {df['revision_count'].min()} (Pad: {df.loc[df['revision_count'].idxmin(), 'pad_id']})")
else:
    print("❌ 未找到任何 Pad 数据")
    df = pd.DataFrame()


🔍 正在查询数据库...

✅ 找到 8 个 Pad



,pad_id,min_revision,max_revision,revision_count
1,room-165,0,2,3
2,room-229,0,4,5
3,room-243,0,0,1
4,room-260,0,0,1
5,room-261,0,0,1
6,room-263,0,3,4
7,room-264,0,0,1
8,room-266,0,45,46



📊 统计信息:
   总 Pad 数: 8
   总版本数: 62
   平均版本数: 7.75
   最多版本: 46 (Pad: room-266)
   最少版本: 1 (Pad: room-243)


## 6. 执行 Pad 内容重建

### 6.1 按照pad_id和起始版本号复原共享文档数据

💡 **使用说明：** 修改下面代码中的参数，然后运行单元格

- `pad_id`: 从上面表格中选择一个 Pad ID
- `start_rev`: 起始版本号
- `end_rev`: 结束版本号
- `batch_size`:每次请求的版本数量

In [6]:
# 📝 在这里修改参数
pad_id = 'room-266'        # 修改为你要查询的 Pad ID
start_rev = 0              # 起始版本号
end_rev = 27              # 结束版本号
use_base64 = True          # 使用 Base64 编码模式（推荐用于大数据量和远程调用）
batch_size = 20            # 每次请求的版本数量（设为 None 则不分批）

# 执行重建
print(f"{'='*80}")
print(f"📝 Pad ID: {pad_id}")
print(f"📊 版本范围: {start_rev} - {end_rev}")
print(f"🔧 编码模式: {'Base64（推荐）' if use_base64 else '标准模式'}")
print(f"📦 批次大小: {'每次请求 ' + str(batch_size) + ' 个版本' if batch_size else '不分批（一次性处理）'}")
print(f"⏰ 开始时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*80}\n")

# 调用函数（自动处理分批逻辑）
rebuild_result = execute_rebuild_script(
    pad_id=pad_id,
    start_rev=start_rev,
    end_rev=end_rev,
    use_base64=use_base64,
    batch_size=batch_size  # 传入 batch_size，函数会自动分批
)

# 显示结果
if rebuild_result.get('success'):
    stats = rebuild_result.get('statistics', {})
    batch_info = rebuild_result.get('batch_info', {})
    
    print(f"{'='*80}")
    print(f"✅ 重建完成！")
    print(f"{'='*80}")
    print(f"   总版本数: {stats.get('total', len(rebuild_result.get('versions', [])))}")
    print(f"   成功: {stats.get('success', 0)}")
    print(f"   失败: {stats.get('failed', 0)}")
    
    if batch_info:
        print(f"   批次数: {batch_info.get('num_batches', 1)}")
        print(f"   批次大小: {batch_info.get('batch_size', 'N/A')}")
    
    print(f"⏰ 完成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"💡 结果已保存到变量 rebuild_result，可以在下面的单元格中查看详细内容")
else:
    print(f"❌ 重建失败: {rebuild_result.get('error', 'Unknown error')}")

📝 Pad ID: room-266
📊 版本范围: 0 - 27
🔧 编码模式: Base64（推荐）
📦 批次大小: 每次请求 20 个版本
⏰ 开始时间: 2026-03-21 14:04:31

📊 总共 28 个版本，将分 2 个批次处理（每批 20 个）

🔄 批次 1/2: 版本 0 - 19
   ✅ 成功: 20, 失败: 0

🔄 批次 2/2: 版本 20 - 27
   ✅ 成功: 8, 失败: 0

✅ 重建完成！
   总版本数: 28
   成功: 28
   失败: 0
   批次数: 2
   批次大小: 20
⏰ 完成时间: 2026-03-21 14:04:31
💡 结果已保存到变量 rebuild_result，可以在下面的单元格中查看详细内容


In [7]:
rebuild_result

{'success': True,
 'pad_id': 'room-266',
 'head_revision': 27,
 'requested_range': {'start': 0, 'end': 27},
 'statistics': {'total': 28, 'success': 28, 'failed': 0},
 'versions': [{'revision': 0,
   'success': True,
   'pad_id': 'room-266',
   'author': 'a.rVEwX679hNTTNivd',
   'timestamp': 1770367830876,
   'formatted_timestamp': '2026-02-06 16:50:30.876',
   'text_length': 228,
   'line_count': 6,
   'change_summary': '1 -> 229 chars',
   'content': 'Welcome to Etherpad!\n\nThis pad text is synchronized as you type, so that everyone viewing this page sees the same text. This allows you to collaborate seamlessly on documents!\n\nGet involved with Etherpad at https://etherpad.org\n',
   'changeset': 'Z:1>6c|5+6c$Welcome to Etherpad!\n\nThis pad text is synchronized as you type, so that everyone viewing this page sees the same text. This allows you to collaborate seamlessly on documents!\n\nGet involved with Etherpad at https://etherpad.org\n',
   'attribs': '|6+6d'},
  {'revision': 1,


In [8]:
# 从 result 中提取版本数据
# result 是一个字典，包含 'versions' 键，里面才是版本列表
if isinstance(rebuild_result, dict):
    versionrange_versions = rebuild_result.get('versions', [])
else:
    # 如果 result 是列表（旧版本兼容）
    versionrange_versions = [v for v in rebuild_result if isinstance(v, dict) and 'revision' in v]

# 直接转 DataFrame
versionrange_rebuild_df = pd.DataFrame(versionrange_versions)

# 设置 Pandas 显示选项
#pd.set_option('display.max_rows', None)  # 显示所有行
#pd.set_option('display.max_columns', None)  # 显示所有列
#pd.set_option('display.max_colwidth', 100)  # 列内容最多显示100字符（可调整）
#pd.set_option('display.width', None)  # 自动调整宽度

# 显示结果
print(f"\n✅ 已转换为 DataFrame，共 {len(versionrange_rebuild_df)} 条版本记录")
if len(versionrange_rebuild_df) > 0:
    print(f"📊 列名: {list(versionrange_rebuild_df.columns)}")
    print(f"📊 版本范围: {versionrange_rebuild_df['revision'].min()} - {versionrange_rebuild_df['revision'].max()}")
else:
    print("⚠️  没有找到版本数据")


✅ 已转换为 DataFrame，共 28 条版本记录
📊 列名: ['revision', 'success', 'pad_id', 'author', 'timestamp', 'formatted_timestamp', 'text_length', 'line_count', 'change_summary', 'content', 'changeset', 'attribs']
📊 版本范围: 0 - 27


In [9]:
versionrange_rebuild_df.head()

,revision,success,pad_id,author,timestamp,formatted_timestamp,text_length,line_count,change_summary,content,changeset,attribs
0,0,True,room-266,a.rVEwX679hNTTNivd,1770367830876,2026-02-06 16:50:30.876,228,6,1 -> 229 chars,Welcome to Etherpad!\n\nThis pad text is synch...,Z:1>6c|5+6c$Welcome to Etherpad!\n\nThis pad t...,|6+6d
1,1,True,room-266,a.rVEwX679hNTTNivd,1770368139358,2026-02-06 16:55:39.358,0,1,229 -> 1 chars,,Z:6d<6c|5-6c$,|1+1
2,2,True,room-266,a.rVEwX679hNTTNivd,1770368277929,2026-02-06 16:57:57.929,130,1,1 -> 131 chars,"For decades, the traditional classroom model—o...","Z:1>3m*0+3m$For decades, the traditional class...",*0+3m|1+1
3,3,True,room-266,a.rVEwX679hNTTNivd,1770368413877,2026-02-06 17:00:13.877,131,1,131 -> 132 chars,"For decades, the traditional classroom model—o...",Z:3n>1=3m*0+1$,*0+3n|1+1
4,4,True,room-266,a.rVEwX679hNTTNivd,1770368414982,2026-02-06 17:00:14.982,133,1,132 -> 134 chars,"For decades, the traditional classroom model—o...",Z:3o>2=3n*0+2$嘻嘻,*0+3p|1+1


### 6.2 按照时间范围复原共享文档数据

💡 **使用说明：** 修改下面代码中的参数，然后运行单元格

- `start_time`: 开始时间
- `end_time`: 结束时间
- `batch_size`:每次请求的版本数量

In [10]:
# ==================== 使用示例 ====================
# 配置参数
start_time = '2026-03-21 13:45:40'  # 开始时间 
end_time = '2026-03-21 20:45:40'    # 结束时间
batch_size = 20  # 批量大小

# API 端点
api_url = f"http://{HTTP_API_CONFIG['host']}:{HTTP_API_CONFIG['port']}/api/rebuild"

# 调用函数查询
pad_versions_with_time = query_pads_by_timerange(start_time, end_time)

🔍 查询时间范围: 2026-03-21 13:45:40 ~ 2026-03-21 20:45:40
📊 时间戳范围: 1774071940000 ~ 1774097140000

🔍 正在查询 Store 数据库...
✅ 找到 12 条记录

📊 正在解析 pad_id 和版本号...
✅ 解析完成，共 2 个不同的 Pad
📋 Pad 版本范围统计:
  - room-229: 版本 1 ~ 4 (共 4 个版本)
  - room-266: 版本 38 ~ 45 (共 8 个版本)


In [11]:
# ==================== 批量调用重建 API ====================
print(f"\n🚀 开始批量重建 (批量大小: {batch_size})...")

all_results = []
total_processed = 0
total_pads = len(pad_versions_with_time)

for idx, (pad_id, revisions) in enumerate(sorted(pad_versions_with_time.items()), 1):
    revisions.sort()
    start_rev = min(revisions)
    end_rev = max(revisions)
    
    print(f"\n[{idx}/{total_pads}] 处理 Pad: {pad_id}")
    print(f"  版本范围: {start_rev} ~ {end_rev}")
    
    # 调用已有的 execute_rebuild_script 函数
    result = execute_rebuild_script(
        pad_id=pad_id,
        start_rev=start_rev,
        end_rev=end_rev,
        use_base64=False,  # 可以根据需要改为 True
        batch_size=batch_size
    )
    
    # 检查结果
    if result.get('success'):
        versions = result.get('versions', [])
        all_results.extend(versions)
        total_processed += len(versions)
        print(f"  ✅ 成功重建 {len(versions)} 个版本")
    else:
        error_msg = result.get('error', '未知错误')
        print(f"  ❌ 重建失败: {error_msg}")

# ==================== 结果统计 ====================
print("" + "="*60)
print("📈 批量重建完成统计")
print("="*60)
print(f"处理的 Pad 数量: {total_pads}")
print(f"成功重建的版本数: {total_processed}")
print(f"平均每个 Pad 的版本数: {total_processed / total_pads if total_pads > 0 else 0:.2f}")


🚀 开始批量重建 (批量大小: 20)...

[1/2] 处理 Pad: room-229
  版本范围: 1 ~ 4
  ✅ 成功重建 4 个版本

[2/2] 处理 Pad: room-266
  版本范围: 38 ~ 45
  ✅ 成功重建 8 个版本
📈 批量重建完成统计
处理的 Pad 数量: 2
成功重建的版本数: 12
平均每个 Pad 的版本数: 6.00


In [12]:
result

{'success': True,
 'pad_id': 'room-266',
 'head_revision': 45,
 'requested_range': {'start': 38, 'end': 45},
 'statistics': {'total': 8, 'success': 8, 'failed': 0},
 'versions': [{'revision': 38,
   'success': True,
   'pad_id': 'room-266',
   'author': 'a.PmwoTwtu6FSe83zV',
   'timestamp': 1774071940641,
   'formatted_timestamp': '2026-03-21 13:45:40.641',
   'text_length': 358,
   'line_count': 5,
   'change_summary': '357 -> 359 chars',
   'content': 'For decades, the traditional classroom model—one teacher, many students, a standardized curriculum—has remained largely unchanged. \n这里删除了好多句 Companies like Carnegie Learning and content platforms like Khan Academy are already implementing such AI-driven tutors, providing a glimpse into a future where education is meticulously customized.\n测试一下增量数据\n\n总结一下  测试',
   'changeset': 'Z:9x>2|4=9q=6*1+2$测试',
   'attribs': '*0|2+9g*0+8*1|2+2*1+8|1+1'},
  {'revision': 39,
   'success': True,
   'pad_id': 'room-266',
   'author': 'a.PmwoTwtu6FSe

In [13]:
# 从 result 中提取版本数据
# result 是一个字典，包含 'versions' 键，里面才是版本列表
if isinstance(result, dict):
    timerange_versions = result.get('versions', [])
else:
    # 如果 result 是列表（旧版本兼容）
    timerange_versions = [v for v in result if isinstance(v, dict) and 'revision' in v]

# 直接转 DataFrame
timerange_rebuild_df = pd.DataFrame(timerange_versions)

# 设置 Pandas 显示选项
#pd.set_option('display.max_rows', None)  # 显示所有行
#pd.set_option('display.max_columns', None)  # 显示所有列
#pd.set_option('display.max_colwidth', 100)  # 列内容最多显示100字符（可调整）
#pd.set_option('display.width', None)  # 自动调整宽度

# 显示结果
print(f"\n✅ 已转换为 DataFrame，共 {len(timerange_rebuild_df)} 条版本记录")
if len(timerange_rebuild_df) > 0:
    print(f"📊 列名: {list(timerange_rebuild_df.columns)}")
    print(f"📊 版本范围: {timerange_rebuild_df['revision'].min()} - {timerange_rebuild_df['revision'].max()}")
else:
    print("⚠️  没有找到版本数据")


✅ 已转换为 DataFrame，共 8 条版本记录
📊 列名: ['revision', 'success', 'pad_id', 'author', 'timestamp', 'formatted_timestamp', 'text_length', 'line_count', 'change_summary', 'content', 'changeset', 'attribs']
📊 版本范围: 38 - 45


In [14]:
timerange_rebuild_df.head()

,revision,success,pad_id,author,timestamp,formatted_timestamp,text_length,line_count,change_summary,content,changeset,attribs
0,38,True,room-266,a.PmwoTwtu6FSe83zV,1774071940641,2026-03-21 13:45:40.641,358,5,357 -> 359 chars,"For decades, the traditional classroom model—o...",Z:9x>2|4=9q=6*1+2$测试,*0|2+9g*0+8*1|2+2*1+8|1+1
1,39,True,room-266,a.PmwoTwtu6FSe83zV,1774071941541,2026-03-21 13:45:41.541,360,5,359 -> 361 chars,"For decades, the traditional classroom model—o...",Z:9z>2|4=9q=8*1+2$一下,*0|2+9g*0+8*1|2+2*1+a|1+1
2,40,True,room-266,a.PmwoTwtu6FSe83zV,1774071942052,2026-03-21 13:45:42.052,361,6,361 -> 362 chars,"For decades, the traditional classroom model—o...",Z:a1>1|4=9q=a*1|1+1$\n,*0|2+9g*0+8*1|3+d|1+1
3,41,True,room-266,a.PmwoTwtu6FSe83zV,1774071943590,2026-03-21 13:45:43.590,362,6,362 -> 363 chars,"For decades, the traditional classroom model—o...",Z:a2>1|5=a1*1+1$证,*0|2+9g*0+8*1|3+d*1+1|1+1
4,42,True,room-266,a.PmwoTwtu6FSe83zV,1774071944137,2026-03-21 13:45:44.137,360,5,363 -> 361 chars,"For decades, the traditional classroom model—o...",Z:a3<2|4=9q=a|1-1-1$,*0|2+9g*0+8*1|2+2*1+a|1+1


## 7. 读取 pad_version_changes 表数据

pad_version_changes通过store数据解析出来，记录变化的数据，add为添加操作，delete为删除操作，从数据库中读取指定 pad 的版本变化记录

In [15]:
def get_pad_version_changes(room_id=None):
    """从数据库中读取指定 pad 的版本变化记录"""
    try:
        connection = pymysql.connect(**DB_CONFIG)
        cursor = connection.cursor(pymysql.cursors.DictCursor)
        
        if room_id:
            query = """SELECT * FROM pad_version_changes WHERE pad_id = %s ORDER BY seq_order ASC"""
            cursor.execute(query, [room_id])
        else:
            query = """SELECT * FROM pad_version_changes ORDER BY pad_id, seq_order ASC"""
            cursor.execute(query)
        
        results = cursor.fetchall()
        
        cursor.close()
        connection.close()
        
        return results
        
    except Exception as e:
        print(f"❌ 数据库查询失败: {e}")
        return []

print("✅ 数据库查询函数定义完成")

✅ 数据库查询函数定义完成


In [16]:
# 读取数据
room_id = "room-229"
df_versions = get_pad_version_changes(room_id=room_id)
df_versions = pd.DataFrame(df_versions)
if df_versions is not None and len(df_versions) > 0:
    df = df_versions[[
    "pad_id",
    "seq_order",
    "behavior",
    "author",
    "content",
    "add_start_time",
    "add_end_time",
    "delete_start_time",
    "delete_end_time"
]]

df_versions.head(5)

,pad_id,seq_order,behavior,author,content,add_start_time,add_end_time,delete_start_time,delete_end_time
0,room-229,1,add,a.rVEwX679hNTTNivd,Welcome to Etherpad!\n\nThis pad text is synch...,2026-02-06 16:51:03.356,2026-02-06 16:51:03.356,None,None
